# Recommender Systems: Zero to Hero

The problem where the metric everyone reports measures the wrong task, and the trivial baseline
usually wins for a reason that turns out to be an artifact of how you measured.

> **Prerequisites:** [`pca_zero_to_hero.ipynb`](pca_zero_to_hero.ipynb) — matrix factorisation
> is the same idea applied to a sparse matrix — and
> [`ml_foundations_zero_to_hero.ipynb`](ml_foundations_zero_to_hero.ipynb) §2 on leakage, since
> §3.2 is about which split is honest. [`time_series_zero_to_hero.ipynb`](time_series_zero_to_hero.ipynb)
> §1.3 is the same "don't shuffle" argument in a different costume.

---

## Why this notebook is different

Recommendation looks like regression — predict the rating a user would give — and almost every
tutorial treats it that way. Measured properly, that framing falls apart:

- **The model with the best RMSE is nearly the worst recommender.** §1.6 measures matrix
  factorisation winning RMSE at **0.782** while its NDCG@10 is **0.0338** — barely above
  random's 0.0166. Ranking and rating prediction are different tasks.
- **Training on the implicit signal instead triples the ranking quality**, with *worse* RMSE.
  §1.7: NDCG 0.0338 → **0.1123**.
- **"Recommend whatever is popular" beats everything** — NDCG **0.2099**. §3.1 then shows
  *why*, and it is not that popularity is a good recommender: turn off the exposure bias in the
  data and popularity **loses** to factorisation (0.0167 against 0.0406).
- **A random train/test split hides the problem you actually have.** §3.2: RMSE 0.654 shuffled
  against **0.782** chronologically, because shuffling puts every user's history on both sides
  and erases the cold-start users a live system meets every day.
- **And the ranking those metrics produce is exactly inverted from what you want.** §3.3 runs
  the recommend-observe-retrain loop under four serving policies: the best-NDCG policy
  concentrates attention onto **20 items out of 800**, and the worst-NDCG policy is the only one
  that widens the catalogue.

## Contents

| Part | What it covers |
|---|---|
| **0. Setup** | Install + imports |
| **1. Theory from zero** | The user-item matrix and sparsity · baselines · neighbourhood methods · **matrix factorisation from scratch** · **why RMSE is the wrong metric** · **implicit feedback** · cold start |
| **2. Worked example** | Building and evaluating a recommender end to end |
| **3. The uncomfortable parts** | **Popularity bias, mechanism proved** · which split is honest · **the feedback loop under four policies** |
| **4. Tough questions** | 12 questions + 3 coding challenges |
| **5. Practice datasets** | 5 datasets with briefs |
| **6. Reading the literature** | The papers behind each section |
| **Appendix** | Recsys-specific errors and a checklist |

## The one-paragraph summary

A recommender starts from a **user-item matrix** that is almost entirely empty — 95%+ missing is
normal. **Collaborative filtering** fills it by exploiting the fact that similar users like
similar things, either through **neighbourhood methods** (find similar items) or **matrix
factorisation** (find latent factors whose product reconstructs the observed entries). The
hard parts are not the algorithm: they are that **your data is missing-not-at-random** because
users only rate what they were shown, that **the metric must measure ranking rather than rating
prediction**, that **new users and items have no history**, and that your model's own
recommendations become tomorrow's training data.

---
# Part 0 - Setup

No extra dependencies. Everything — matrix factorisation, item-item similarity, the ranking
metrics — is implemented from scratch in a few lines each, which is also the best way to see
what the libraries are doing.

In [ ]:
# ---------------------------------------------------------------------------
# One-time setup. Only missing packages are installed, so re-running is cheap.
# ---------------------------------------------------------------------------
import importlib.util
import subprocess
import sys

REQUIRED = [
    ("numpy", "numpy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"),
    ("scipy", "scipy"), ("sklearn", "scikit-learn"),
]
missing = [pip for mod, pip in REQUIRED if importlib.util.find_spec(mod) is None]
if missing:
    print("installing:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("done")
else:
    print("all packages present")

In [ ]:
# ---------------------------------------------------------------------------
# Every import this notebook uses.
# ---------------------------------------------------------------------------
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sp

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 40)
plt.rcParams["figure.figsize"] = (9, 3.4)
plt.rcParams["figure.dpi"] = 110

print("ready | numpy", np.__version__, "| pandas", pd.__version__)

---
# Part 1 - Theory from zero

1. The user-item matrix, and how empty it is
2. The data we will use, and why it is synthetic
3. Baselines — and the one that is hard to beat
4. Neighbourhood methods: item-item collaborative filtering
5. **Matrix factorisation from scratch**
6. **Why RMSE is the wrong metric**
7. **Implicit feedback: training on the right task**
8. Cold start

## 1.1 The user-item matrix

Everything starts with one object: a matrix $R$ with users as rows, items as columns, and
ratings (or clicks, purchases, watch time) in the cells.

Two facts define the whole field:

1. **It is almost entirely empty.** A user rates a few dozen of a million items. Densities of
   0.1%–5% are normal, and the task is to fill in the rest.
2. **The missing entries are not missing at random.** A user has not rated a film because they
   never saw it — and *what they saw* was chosen by popularity, marketing, and the previous
   recommender. §3.1 shows this being the single most consequential fact in the notebook.

The second point is what separates recommendation from ordinary supervised learning, where we
usually assume the rows we have are a fair sample of the rows we care about.

In [ ]:
# A tiny matrix, printed in full, to make the shape concrete.
demo = pd.DataFrame(
    [[5, np.nan, 4, np.nan, 1],
     [np.nan, 4, np.nan, np.nan, 2],
     [4, 5, np.nan, 3, np.nan],
     [np.nan, np.nan, 2, np.nan, 5],
     [1, np.nan, np.nan, 4, np.nan]],
    index=[f"user {i}" for i in range(5)],
    columns=[f"item {j}" for j in range(5)])
print(demo.to_string(na_rep="  ."))
print(f"\nobserved cells: {demo.notna().sum().sum()} of {demo.size} "
      f"({100*demo.notna().sum().sum()/demo.size:.0f}%)")
print()
print("Every '.' is a question, not a zero. The whole task is deciding which of them would")
print("have been high - and, crucially, WHY each one is blank (1.1, 3.1).")
print()
print("Two families of answer:")
print("  CONTENT-BASED     - describe items by their attributes (genre, director, text) and")
print("                      recommend items similar to what this user liked. Works for new")
print("                      items; cannot discover taste the attributes do not capture.")
print("  COLLABORATIVE     - use only the interaction matrix: users who agreed before will")
print("                      agree again. Needs no item metadata, finds surprising things,")
print("                      and fails completely for a brand-new item (1.8).")
print()
print("This notebook is about the collaborative family, which is what 'recommender system'")
print("usually means. Real systems are hybrids.")

## 1.2 The data, and why it is synthetic

Every other notebook in this series uses real data. This one uses a **generated** dataset, and
the reason is worth stating plainly.

**The practical reason:** the canonical dataset here is MovieLens, and at the time of writing
the GroupLens download fails — its TLS certificate has expired, so `https://files.grouplens.org`
cannot be fetched. A notebook whose first cell may not run is not much use. §5 gives the loader
for when it is fixed.

**The better reason:** the two most important claims in this notebook can only be *proved* with
data whose truth you control.

- §1.5 checks whether factorisation recovers the **true latent factors**. On real data you
  cannot check that, because you do not know them.
- §3.1 proves that popularity's apparent dominance is caused by **exposure bias** by turning
  that bias off and watching the result reverse. No real dataset has that switch.

So the generator below is built to be realistic in the four ways that matter: low-rank
preferences, power-law item popularity, power-law user activity, and — the important one —
**observation probability that depends on popularity**, which is what makes the data
missing-not-at-random.

In [ ]:
def make_interactions(n_users=1500, n_items=800, n_factors=6, density=0.045,
                      pop_exponent=1.0, seed=RANDOM_STATE):
    """A user-item ratings dataset with KNOWN latent structure.

    pop_exponent controls how strongly item popularity drives WHICH cells are observed.
    At 0 every user-item pair is equally likely to be rated (unrealistic, but it is the
    control condition for 3.1). At 1 the data is missing-not-at-random, as real data is.
    """
    rng = np.random.default_rng(seed)

    U = rng.normal(0, 1, (n_users, n_factors))          # true user taste
    V = rng.normal(0, 1, (n_items, n_factors))          # true item character
    user_bias = rng.normal(0, 0.35, n_users)            # some users rate generously
    item_bias = rng.normal(0, 0.50, n_items)            # some items are just better
    mu = 3.5
    true_score = mu + user_bias[:, None] + item_bias[None, :] + (U @ V.T) / np.sqrt(n_factors)

    item_pop = rng.pareto(1.2, n_items) + 1.0           # a few blockbusters, a long tail
    item_pop /= item_pop.sum()
    user_act = rng.pareto(1.4, n_users) + 1.0           # a few heavy users
    user_act /= user_act.sum()

    n_obs = int(n_users * n_items * density)
    p = np.outer(user_act, item_pop ** pop_exponent)    # <- the MNAR mechanism
    p /= p.sum()
    flat = rng.choice(n_users * n_items, size=n_obs, replace=False, p=p.ravel())
    users, items = np.unravel_index(flat, (n_users, n_items))

    noisy = true_score[users, items] + rng.normal(0, 0.5, n_obs)
    ratings = np.clip(np.round(noisy * 2) / 2, 1.0, 5.0)          # half-stars, 1..5

    # Users JOIN at different times, so a chronological split genuinely contains
    # users the model has never seen - exactly as a live system does (3.2).
    join = rng.uniform(0.0, 0.75, n_users)
    ts = (join[users] + rng.uniform(0.0, 0.25, n_obs)) * 100_000

    d = pd.DataFrame({"user": users, "item": items, "rating": ratings, "ts": ts})
    return (d.sort_values("ts").reset_index(drop=True),
            {"true_score": true_score, "user_bias": user_bias, "item_bias": item_bias,
             "U": U, "V": V, "n_users": n_users, "n_items": n_items})


ratings_df, TRUTH = make_interactions()
N_USERS, N_ITEMS = TRUTH["n_users"], TRUTH["n_items"]

print(f"interactions : {len(ratings_df):,}")
print(f"users        : {ratings_df['user'].nunique():,}")
print(f"items        : {ratings_df['item'].nunique():,}")
print(f"density      : {100*len(ratings_df)/(N_USERS*N_ITEMS):.2f}% of the matrix observed")
print(f"             : so {100 - 100*len(ratings_df)/(N_USERS*N_ITEMS):.2f}% is the thing "
      f"we are trying to predict\n")

vc = ratings_df["rating"].value_counts(normalize=True).sort_index()
print("rating distribution:")
for r, p in vc.items():
    print(f"  {r:>3.1f}  {100*p:>4.1f}%  {'#' * int(70*p/vc.max())}")

pop = ratings_df["item"].value_counts()
act = ratings_df["user"].value_counts()
print(f"\nitem popularity : top item {pop.iloc[0]:,} ratings, median {int(pop.median())}, "
      f"least {pop.iloc[-1]}")
print(f"                : the top 10% of items hold "
      f"{100*pop.head(len(pop)//10).sum()/len(ratings_df):.0f}% of all ratings")
print(f"user activity   : most active {act.iloc[0]:,}, median {int(act.median())}, "
      f"least {act.iloc[-1]}")
print()
print("Those two skews are not decoration. The long tail of items is what makes 'recommend")
print("the popular thing' a strong baseline (1.3), and the long tail of users is what makes")
print("cold start a permanent condition rather than an edge case (1.8).")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))

axes[0].loglog(np.arange(1, len(pop) + 1), pop.to_numpy(), lw=1.2)
axes[0].set(title="Item popularity (log-log)", xlabel="item rank", ylabel="ratings")

axes[1].loglog(np.arange(1, len(act) + 1), act.to_numpy(), lw=1.2, color="crimson")
axes[1].set(title="User activity (log-log)", xlabel="user rank", ylabel="ratings")

sub = ratings_df[(ratings_df["user"] < 60) & (ratings_df["item"] < 120)]
grid = np.full((60, 120), np.nan)
grid[sub["user"].to_numpy(), sub["item"].to_numpy()] = sub["rating"].to_numpy()
axes[2].imshow(grid, aspect="auto", cmap="viridis", interpolation="nearest")
axes[2].set(title="A corner of the matrix (blank = missing)", xlabel="item", ylabel="user")

for ax in axes[:2]:
    ax.grid(alpha=0.3, which="both")
fig.tight_layout(); plt.show()

print("A straight line on a log-log plot is a power law: a handful of items get almost all")
print("the attention, and most items get almost none. The right-hand panel is what the model")
print("actually sees - mostly holes.")

## 1.3 Baselines

As always, baselines first. For recommendation there are three, and the third is the one that
should worry you.

In [ ]:
CUT = int(len(ratings_df) * 0.8)
train = ratings_df.iloc[:CUT].copy()
test = ratings_df.iloc[CUT:].copy()
print(f"chronological split: train {len(train):,}, test {len(test):,}")
print(f"users in test never seen in training: "
      f"{len(set(test['user']) - set(train['user']))}")
print(f"items in test never seen in training: "
      f"{len(set(test['item']) - set(train['item']))}\n")

MU = train["rating"].mean()
item_mean = train.groupby("item")["rating"].mean()
user_mean = train.groupby("user")["rating"].mean()
item_count = train.groupby("item").size()

ITEM_MEAN = np.full(N_ITEMS, MU); ITEM_MEAN[item_mean.index.to_numpy()] = item_mean.to_numpy()
USER_MEAN = np.full(N_USERS, MU); USER_MEAN[user_mean.index.to_numpy()] = user_mean.to_numpy()
ITEM_POP = np.zeros(N_ITEMS); ITEM_POP[item_count.index.to_numpy()] = item_count.to_numpy()


def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)))


u_te = test["user"].to_numpy(); i_te = test["item"].to_numpy(); r_te = test["rating"].to_numpy()
print(f"{'baseline':<34} {'test RMSE':>11}")
print("-" * 48)
for name, pred in [
    ("predict the global mean", np.full(len(test), MU)),
    ("predict the item's mean rating", ITEM_MEAN[i_te]),
    ("predict the user's mean rating", USER_MEAN[u_te]),
    ("user mean + item offset", USER_MEAN[u_te] + (ITEM_MEAN[i_te] - MU)),
]:
    print(f"{name:<34} {rmse(r_te, pred):>11.4f}")

print()
print("'User mean plus item offset' is the baseline every recommender must beat, and it is")
print("only two group-bys. A large share of the variance in ratings is simply that some")
print("users are generous and some items are good.")
print()
print("But notice something missing from this table: none of these baselines RANKS anything.")
print("They all predict a number. Whether that is the right thing to measure is 1.6.")

## 1.4 Neighbourhood methods

The oldest collaborative approach, and still the one that is easiest to explain to a
stakeholder: **"because you liked X, and people who liked X also liked Y."**

**Item-item** is the standard form (Sarwar et al., 2001) — compute a similarity between every
pair of items from their rating vectors, then score an unseen item by how similar it is to the
things this user already liked. It is preferred over user-user because item similarities are
more stable over time and there are usually fewer items than users.

Two details separate a working implementation from a broken one: **mean-centre** each user's
ratings first (so "generous rater" does not look like "similar taste"), and **shrink**
similarities computed from few co-ratings.

In [ ]:
def item_item_similarity(train, n_items, shrink=10.0):
    """Cosine similarity between mean-centred item rating vectors, shrunk by support."""
    u = train["user"].to_numpy(); i = train["item"].to_numpy()
    r = train["rating"].to_numpy()
    um = train.groupby("user")["rating"].mean()
    centred = r - um.reindex(u).to_numpy()                 # remove each user's own scale

    M = sp.csr_matrix((centred, (i, u)), shape=(n_items, int(u.max()) + 1))
    norms = np.sqrt(np.asarray(M.multiply(M).sum(axis=1)).ravel())
    norms[norms == 0] = 1e-9
    S = np.asarray((M @ M.T).todense()) / np.outer(norms, norms)

    counts = (M != 0).astype(float)                        # how many users rated BOTH
    co = np.asarray((counts @ counts.T).todense())
    S *= co / (co + shrink)                                # trust similarity less on thin data
    np.fill_diagonal(S, 0.0)
    return S


t0 = time.time()
SIM = item_item_similarity(train, N_ITEMS)
print(f"item-item similarity matrix {SIM.shape} in {time.time()-t0:.2f}s\n")

off = SIM[~np.eye(N_ITEMS, dtype=bool)]
print(f"similarities: mean {off.mean():+.4f}, max {off.max():.4f}, min {off.min():.4f}")
print(f"pairs with similarity > 0.3: {(off > 0.3).sum() // 2:,}")

# Do similar items really share latent character? Only checkable on synthetic data.
V = TRUTH["V"]
Vn = V / np.linalg.norm(V, axis=1, keepdims=True)
true_sim = Vn @ Vn.T
mask = ~np.eye(N_ITEMS, dtype=bool)
keep = (SIM != 0) & mask
print(f"\ncorrelation between the LEARNED similarity and the TRUE latent similarity: "
      f"{np.corrcoef(SIM[keep], true_sim[keep])[0,1]:.4f}")
print("  (positive, but far from 1 - with 4.5% density there is not much to work with)")

probe = int(np.asarray(ITEM_POP).argmax())
nb = np.argsort(-SIM[probe])[:5]
print(f"\nthe 5 items most similar to item {probe} (the most-rated item):")
for j in nb:
    print(f"  item {j:>3}  similarity {SIM[probe, j]:+.4f}  "
          f"({int(ITEM_POP[j])} ratings, mean {ITEM_MEAN[j]:.2f})")

In [ ]:
# Score every unseen item for a user: weight their past ratings by item similarity.
USER_PROFILE = np.asarray(sp.csr_matrix(
    (train["rating"].to_numpy() - USER_MEAN[train["user"].to_numpy()],
     (train["user"].to_numpy(), train["item"].to_numpy())),
    shape=(N_USERS, N_ITEMS)).todense())

ITEM_ITEM_SCORES = USER_PROFILE @ SIM         # (users x items)

print(f"scores matrix {ITEM_ITEM_SCORES.shape}\n")
who = int(train["user"].value_counts().index[0])
liked = train[(train["user"] == who) & (train["rating"] >= 4.5)]["item"].to_numpy()[:5]
print(f"user {who} rated {int((train['user']==who).sum())} items; some they loved: "
      f"{liked.tolist()}")
seen = set(train[train["user"] == who]["item"].tolist())
cand = np.array([j for j in range(N_ITEMS) if j not in seen])
top = cand[np.argsort(-ITEM_ITEM_SCORES[who, cand])[:5]]
print(f"top-5 recommendations for user {who}: {top.tolist()}")
print()
print("The output is a RANKING, not a rating. That is what a recommender is actually for, and")
print("it is why 1.6 has to change the metric.")
print()
print("Cost note: the similarity matrix is items x items. With 800 items that is trivial;")
print("with a million items it is 10^12 entries and you need approximate nearest neighbours")
print("(NB-07 section 3.2) or a factorisation (1.5) instead.")

## 1.5 Matrix factorisation

The idea that won the Netflix Prize and still underpins most production systems. Approximate
the ratings matrix as a product of two thin matrices:

$$ R \approx \mu + b_u + b_i + P Q^\top $$

Each user gets a short vector $p_u$ (their taste), each item a vector $q_i$ (its character), and
the predicted rating is their dot product plus bias terms. With $k$ factors you replace
$n_u \times n_i$ unknowns with $(n_u + n_i) \times k$ — the same compression argument as PCA
(NB-09), applied to a matrix that is mostly missing.

It is fitted by **alternating least squares**: hold $Q$ fixed and solve for $P$ (a ridge
regression per user), then hold $P$ fixed and solve for $Q$. Each step is a closed form, and
the loss decreases every time.

In [ ]:
def als(train, n_users, n_items, k=6, iters=12, reg=0.1, seed=RANDOM_STATE):
    """Biased matrix factorisation by alternating least squares, from scratch."""
    rng = np.random.default_rng(seed)
    mu = train["rating"].mean()
    u = train["user"].to_numpy(); i = train["item"].to_numpy()
    r = train["rating"].to_numpy() - mu

    P = rng.normal(0, 0.05, (n_users, k))
    Q = rng.normal(0, 0.05, (n_items, k))
    bu = np.zeros(n_users); bi = np.zeros(n_items)

    for _ in range(iters):
        # --- biases: closed form given everything else
        resid = r - (bi[i] + np.einsum("ij,ij->i", P[u], Q[i]))
        bu = np.bincount(u, weights=resid, minlength=n_users) / \
             (np.bincount(u, minlength=n_users) + reg * 10)
        resid = r - (bu[u] + np.einsum("ij,ij->i", P[u], Q[i]))
        bi = np.bincount(i, weights=resid, minlength=n_items) / \
             (np.bincount(i, minlength=n_items) + reg * 10)

        # --- factors: one small ridge solve per user, then per item
        target = r - bu[u] - bi[i]
        Rc = sp.csr_matrix((target, (u, i)), shape=(n_users, n_items))
        for row in range(n_users):
            s, e = Rc.indptr[row], Rc.indptr[row + 1]
            if e == s:
                continue
            Qi = Q[Rc.indices[s:e]]
            P[row] = np.linalg.solve(Qi.T @ Qi + reg * np.eye(k), Qi.T @ Rc.data[s:e])
        Rct = Rc.T.tocsr()
        for col in range(n_items):
            s, e = Rct.indptr[col], Rct.indptr[col + 1]
            if e == s:
                continue
            Pu = P[Rct.indices[s:e]]
            Q[col] = np.linalg.solve(Pu.T @ Pu + reg * np.eye(k), Pu.T @ Rct.data[s:e])
    return {"P": P, "Q": Q, "bu": bu, "bi": bi, "mu": mu}


def mf_predict(m, u, i):
    return m["mu"] + m["bu"][u] + m["bi"][i] + np.einsum("ij,ij->i", m["P"][u], m["Q"][i])


t0 = time.time()
MF = als(train, N_USERS, N_ITEMS, k=6, iters=12)
print(f"ALS fitted in {time.time()-t0:.1f}s\n")

pred_mf = mf_predict(MF, u_te, i_te)
print(f"{'model':<34} {'test RMSE':>11}")
print("-" * 48)
print(f"{'user mean + item offset':<34} "
      f"{rmse(r_te, USER_MEAN[u_te] + (ITEM_MEAN[i_te] - MU)):>11.4f}")
print(f"{'matrix factorisation (k=6)':<34} {rmse(r_te, pred_mf):>11.4f}")

In [ ]:
# Did it recover the TRUE structure? This is the check you cannot run on real data.
true_te = TRUTH["true_score"][u_te, i_te]
seen_u = np.unique(train["user"]); seen_i = np.unique(train["item"])

print("Checking the model against the ground truth the generator used:\n")
print(f"  correlation of prediction with the TRUE noiseless score : "
      f"{np.corrcoef(pred_mf, true_te)[0,1]:.4f}")
print(f"  recovered ITEM bias vs the true item bias               : "
      f"{np.corrcoef(MF['bi'][seen_i], TRUTH['item_bias'][seen_i])[0,1]:.4f}")
print(f"  recovered USER bias vs the true user bias               : "
      f"{np.corrcoef(MF['bu'][seen_u], TRUTH['user_bias'][seen_u])[0,1]:.4f}")
print(f"\n  noise deliberately added to every rating: sd 0.50")
print(f"  so an RMSE near 0.5 is the floor - no model can do better")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].scatter(TRUTH["item_bias"][seen_i], MF["bi"][seen_i], s=6, alpha=0.5)
axes[0].set(xlabel="true item bias", ylabel="recovered item bias",
            title="Item biases are recovered well")
axes[1].scatter(TRUTH["user_bias"][seen_u], MF["bu"][seen_u], s=6, alpha=0.5, color="crimson")
axes[1].set(xlabel="true user bias", ylabel="recovered user bias",
            title="User biases too")
for ax in axes:
    ax.grid(alpha=0.3)
fig.tight_layout(); plt.show()

print("The individual FACTORS are not directly comparable - factorisation is unique only up")
print("to rotation, exactly as PCA's components are unique only up to sign (NB-09 section 1.3).")
print("The biases are identifiable, and they check out.")

print("\nHow many factors? More capacity is not automatically better on a sparse matrix:")
print(f"  {'k':>4} {'test RMSE':>11} {'parameters':>12}")
print("  " + "-" * 30)
for k in [1, 2, 4, 6, 10, 20, 40]:
    m = als(train, N_USERS, N_ITEMS, k=k, iters=10)
    print(f"  {k:>4} {rmse(r_te, mf_predict(m, u_te, i_te)):>11.4f} "
          f"{(N_USERS + N_ITEMS) * k:>12,}")
print("\n  The true number of factors used by the generator was 6.")

## 1.6 Why RMSE is the wrong metric

Here is the section that should change how you read every recommender paper and blog post.

RMSE measures how well you predict **the rating a user gave to an item they chose to rate**.
A recommender's job is different: **pick a handful of items, out of everything, that this user
would want**. Those are not the same task, and optimising the first does not deliver the second.

The metrics that measure the real task:

- **precision@k** — of the $k$ items you recommended, what fraction were relevant?
- **recall@k** — of the items the user actually liked, what fraction did you surface?
- **NDCG@k** — like precision, but rewards putting the good items *nearer the top*.

In [ ]:
def ranking_metrics(train, test, score_fn, n_items, k=10, relevant_at=4.0):
    """For each test user: rank every item they have NOT already seen, score the top k."""
    seen = train.groupby("user")["item"].apply(lambda s: set(s)).to_dict()
    liked = (test[test["rating"] >= relevant_at]
             .groupby("user")["item"].apply(lambda s: set(s)).to_dict())
    all_items = np.arange(n_items)
    P, R, N = [], [], []
    for uu, relevant in liked.items():
        if not relevant:
            continue
        mask = np.ones(n_items, dtype=bool)
        already = seen.get(uu)
        if already:
            mask[list(already)] = False
        cand = all_items[mask]
        if len(cand) == 0:
            continue
        top = cand[np.argsort(-score_fn(np.full(len(cand), uu), cand))[:k]]
        gains = np.array([1.0 if t in relevant else 0.0 for t in top])
        dcg = np.sum(gains / np.log2(np.arange(2, k + 2)))
        best = min(k, len(relevant))
        idcg = np.sum(np.ones(best) / np.log2(np.arange(2, best + 2)))
        P.append(gains.sum() / k)
        R.append(gains.sum() / len(relevant))
        N.append(dcg / idcg if idcg > 0 else 0.0)
    return float(np.mean(P)), float(np.mean(R)), float(np.mean(N)), len(P)


rng_demo = np.random.default_rng(RANDOM_STATE)
SCORERS = {
    "random": lambda u, i: rng_demo.random(len(i)),
    "most popular": lambda u, i: ITEM_POP[i],
    "item mean rating": lambda u, i: ITEM_MEAN[i],
    "item-item CF": lambda u, i: ITEM_ITEM_SCORES[u, i],
    "matrix factorisation": lambda u, i: mf_predict(MF, u, i),
}

print(f"{'model':<26} {'test RMSE':>11} {'P@10':>9} {'R@10':>9} {'NDCG@10':>10}")
print("-" * 70)
for name, fn in SCORERS.items():
    try:
        p = fn(u_te, i_te)
        rm = rmse(r_te, p)
        rm_s = f"{rm:>11.4f}" if rm < 50 else f"{'n/a':>11}"
    except Exception:
        rm_s = f"{'n/a':>11}"
    pk, rk, nd, n_eval = ranking_metrics(train, test, fn, N_ITEMS, k=10)
    print(f"{name:<26} {rm_s} {pk:>9.4f} {rk:>9.4f} {nd:>10.4f}")
print(f"\nevaluated on {n_eval} users with at least one liked item in the test period")

print()
print("Read the first column against the last one. They rank the models differently.")
print()
print("  MATRIX FACTORISATION has the BEST RMSE by a wide margin - and an NDCG barely above")
print("  RANDOM. It is an excellent rating predictor and a poor recommender.")
print()
print("  MOST POPULAR has no meaningful RMSE at all - its score is a count, not a rating -")
print("  and the best NDCG of everything here.")
print()
print("Why: RMSE is computed only on items the user ALREADY CHOSE to rate. It never asks the")
print("question a recommender faces, which is 'out of 800 items, which 10?'. A model can be")
print("excellent at the first and useless at the second.")
print()
print("This is not a subtlety. The Netflix Prize was scored on RMSE, and Netflix has said the")
print("winning ensemble was never fully deployed (Q3).")

## 1.7 Implicit feedback: train on the task you are measured on

If the problem is that we trained a rating predictor and measured a ranker, the fix is to train
a ranker.

**Implicit feedback** (Hu, Koren & Volinsky, 2008) reframes the data. Instead of "this user gave
this item 4.5 stars", the signal is **"this user interacted with this item at all"** — every
observed cell is a **positive**, every unobserved cell is a weak **negative**, and the rating
becomes a *confidence* weight rather than a target.

That matches what most real systems actually have — clicks, plays, purchases — and, crucially,
it makes the model learn what users *choose*, not just how they score what they chose.

In [ ]:
def implicit_als(train, n_users, n_items, k=16, iters=10, reg=0.1, alpha=20.0,
                 seed=RANDOM_STATE):
    """Hu, Koren & Volinsky (2008). Fits ALL cells: observed = 1 with high confidence,
    unobserved = 0 with confidence 1. The trick is that the unobserved term factorises,
    so you never materialise the full matrix."""
    rng = np.random.default_rng(seed)
    u = train["user"].to_numpy(); i = train["item"].to_numpy()
    conf = 1.0 + alpha * (train["rating"].to_numpy() / 5.0)

    C = sp.csr_matrix((conf, (u, i)), shape=(n_users, n_items))
    Ct = C.T.tocsr()
    X = rng.normal(0, 0.05, (n_users, k))
    Y = rng.normal(0, 0.05, (n_items, k))
    I = np.eye(k)

    for _ in range(iters):
        YtY = Y.T @ Y                                    # the "all zeros" term, once
        for row in range(n_users):
            s, e = C.indptr[row], C.indptr[row + 1]
            if e == s:
                continue
            idx, c = C.indices[s:e], C.data[s:e]
            Yi = Y[idx]
            X[row] = np.linalg.solve(YtY + (Yi * (c - 1)[:, None]).T @ Yi + reg * I,
                                     (Yi * c[:, None]).sum(axis=0))
        XtX = X.T @ X
        for col in range(n_items):
            s, e = Ct.indptr[col], Ct.indptr[col + 1]
            if e == s:
                continue
            idx, c = Ct.indices[s:e], Ct.data[s:e]
            Xu = X[idx]
            Y[col] = np.linalg.solve(XtX + (Xu * (c - 1)[:, None]).T @ Xu + reg * I,
                                     (Xu * c[:, None]).sum(axis=0))
    return {"X": X, "Y": Y}


t0 = time.time()
IMF = implicit_als(train, N_USERS, N_ITEMS, k=16, iters=10)
print(f"implicit ALS fitted in {time.time()-t0:.1f}s\n")

SCORERS["implicit ALS"] = lambda u, i: np.einsum("ij,ij->i", IMF["X"][u], IMF["Y"][i])

print(f"{'model':<26} {'test RMSE':>11} {'P@10':>9} {'R@10':>9} {'NDCG@10':>10}")
print("-" * 70)
for name, fn in SCORERS.items():
    try:
        p = fn(u_te, i_te)
        rm = rmse(r_te, p)
        rm_s = f"{rm:>11.4f}" if rm < 50 else f"{'n/a':>11}"
    except Exception:
        rm_s = f"{'n/a':>11}"
    pk, rk, nd, _ = ranking_metrics(train, test, fn, N_ITEMS, k=10)
    print(f"{name:<26} {rm_s} {pk:>9.4f} {rk:>9.4f} {nd:>10.4f}")

print()
print("The implicit model has a MUCH worse RMSE - it is not predicting ratings at all, so the")
print("number is meaningless - and roughly triples the explicit model's NDCG.")
print()
print("That is the lesson of 1.6 and 1.7 together: the biggest single improvement available")
print("was not a better model or more factors. It was training on the task being measured.")
print()
print("Two honest caveats before you take the win. It still does not beat 'most popular' -")
print("3.1 shows exactly what is going on there. And it does not beat plain item-item CF")
print("either, which is a reminder that in this field the simple baseline is routinely")
print("competitive; Q11's reading list has a whole paper about that.")

## 1.8 Cold start

A collaborative model knows a user only through their history. With no history there is no
$p_u$ to look up, and the model has nothing to say.

This is not an edge case. Because user activity follows a power law (§1.2), *most* users have
very little history at any moment, and every user starts with none.

In [ ]:
history = train.groupby("user").size()
te = test.copy()
te["n_train"] = te["user"].map(history).fillna(0).astype(int)
te["pred"] = mf_predict(MF, te["user"].to_numpy(), te["item"].to_numpy())
te["sq_err"] = (te["rating"] - te["pred"]) ** 2

print(f"{'ratings this user had in training':>34} {'test rows':>11} {'RMSE':>9}")
print("-" * 58)
for lo, hi, label in [(0, 0, "0 - never seen before"), (1, 4, "1-4"), (5, 19, "5-19"),
                      (20, 49, "20-49"), (50, 10**9, "50+")]:
    s = te[(te["n_train"] >= lo) & (te["n_train"] <= hi)]
    if len(s) == 0:
        continue
    print(f"{label:>34} {len(s):>11,} {np.sqrt(s['sq_err'].mean()):>9.4f}")

cold_users = set(test["user"]) - set(train["user"])
print(f"\nusers appearing in test but never in training: {len(cold_users)}")
print(f"share of test rows from users with fewer than 5 prior ratings: "
      f"{(te['n_train'] < 5).mean():.1%}")

plt.plot(*zip(*[(n, np.sqrt(te[te["n_train"] == n]["sq_err"].mean()))
                for n in range(0, 40)
                if (te["n_train"] == n).sum() >= 20]), marker="o", ms=3)
plt.xlabel("ratings the user had in training"); plt.ylabel("test RMSE")
plt.title("Error falls as a user accumulates history")
plt.grid(alpha=0.3); plt.show()

print()
print("Error is roughly 60% higher for a brand-new user than for a well-known one, and it")
print("improves steeply over the first handful of ratings before flattening.")
print()
print("What to do about it - none of these is a model change:")
print("  NEW USER  - recommend popular items (which is why 1.6's baseline is strong), ask a")
print("              few onboarding questions, or use whatever you know: country, device,")
print("              referrer.")
print("  NEW ITEM  - fall back to CONTENT features (1.1). This is where a pure collaborative")
print("              model has literally nothing, and why real systems are hybrids.")
print("  IN GENERAL- blend the personalised score toward the popularity prior with a weight")
print("              that grows with the user's history. Q7 works through the arithmetic.")

---
# Part 2 - Worked example: building a recommender end to end

Part 1 built the pieces. This part assembles them the way you would for a real system, and
reports the things a real evaluation has to report.

## 2.1 A hybrid scorer

No production system uses one model. The standard shape is a **blend**: a personalised score,
backed off toward a popularity prior when the personalisation cannot be trusted (§1.8).

In [ ]:
def hybrid_scorer(user_history_counts, personal_fn, prior_fn, k_shrink=10.0):
    """Blend a personalised score toward a popularity prior.

    weight = n / (n + k_shrink), so a user with no history gets pure prior and a user
    with plenty gets almost pure personalisation. This is the same shrinkage idea as
    the similarity shrink in 1.4.
    """
    def score(u, i):
        n = user_history_counts[u]
        w = n / (n + k_shrink)
        p = personal_fn(u, i)
        q = prior_fn(u, i)
        # put both on a comparable scale before mixing
        p = (p - p.mean()) / (p.std() + 1e-9)
        q = (q - q.mean()) / (q.std() + 1e-9)
        return w * p + (1 - w) * q
    return score


hist_counts = np.zeros(N_USERS)
hc = train.groupby("user").size()
hist_counts[hc.index.to_numpy()] = hc.to_numpy()

implicit_fn = lambda u, i: np.einsum("ij,ij->i", IMF["X"][u], IMF["Y"][i])
popular_fn = lambda u, i: ITEM_POP[i]
HYBRID = hybrid_scorer(hist_counts, implicit_fn, popular_fn, k_shrink=10.0)

print(f"{'model':<30} {'P@10':>9} {'R@10':>9} {'NDCG@10':>10}")
print("-" * 62)
for name, fn in [("most popular (prior alone)", popular_fn),
                 ("implicit ALS (personal alone)", implicit_fn),
                 ("hybrid: shrink to the prior", HYBRID)]:
    pk, rk, nd, _ = ranking_metrics(train, test, fn, N_ITEMS, k=10)
    print(f"{name:<30} {pk:>9.4f} {rk:>9.4f} {nd:>10.4f}")

print("\nblend weight by user history:")
for n in [0, 1, 5, 10, 25, 100]:
    print(f"  {n:>4} prior ratings -> {n/(n+10):.0%} personalised, {10/(n+10):.0%} popularity")
print()
print("Note what the hybrid did NOT do: it did not beat the popularity prior it was blended")
print("with. It lands a hair below it and a long way above the personalised model on its own.")
print("On this metric, on this data, adding personalisation to popularity buys nothing - and")
print("2.2 is about why that verdict is the metric's fault rather than the model's.")
print()
print("The hybrid is still the shape to remember, because it degrades gracefully: a brand-new")
print("user gets sensible popular recommendations rather than noise, a heavy user gets")
print("personalised ones, and there is no special-casing anywhere in the code.")

## 2.2 Evaluate at several k, and report coverage

A single precision@10 hides two things a product owner will ask about immediately: how the
quality changes with list length, and **how much of the catalogue you ever recommend**.

In [ ]:
print(f"{'model':<30} " + " ".join(f"{'NDCG@'+str(k):>9}" for k in [5, 10, 20, 50]))
print("-" * 72)
for name, fn in [("most popular", popular_fn),
                 ("item-item CF", lambda u, i: ITEM_ITEM_SCORES[u, i]),
                 ("matrix factorisation", lambda u, i: mf_predict(MF, u, i)),
                 ("implicit ALS", implicit_fn),
                 ("hybrid", HYBRID)]:
    vals = []
    for k in [5, 10, 20, 50]:
        _, _, nd, _ = ranking_metrics(train, test, fn, N_ITEMS, k=k)
        vals.append(nd)
    print(f"{name:<30} " + " ".join(f"{v:>9.4f}" for v in vals))


def catalogue_coverage(train, test, score_fn, n_items, k=10):
    """What fraction of the catalogue ever appears in anyone's top-k?"""
    seen = train.groupby("user")["item"].apply(lambda s: set(s)).to_dict()
    users = test["user"].unique()
    recommended = set()
    all_items = np.arange(n_items)
    for uu in users:
        mask = np.ones(n_items, dtype=bool)
        already = seen.get(uu)
        if already:
            mask[list(already)] = False
        cand = all_items[mask]
        top = cand[np.argsort(-score_fn(np.full(len(cand), uu), cand))[:k]]
        recommended.update(top.tolist())
    return len(recommended) / n_items


print(f"\n{'model':<30} {'catalogue coverage @10':>24}")
print("-" * 58)
for name, fn in [("most popular", popular_fn),
                 ("item-item CF", lambda u, i: ITEM_ITEM_SCORES[u, i]),
                 ("matrix factorisation", lambda u, i: mf_predict(MF, u, i)),
                 ("implicit ALS", implicit_fn),
                 ("hybrid", HYBRID)]:
    print(f"{name:<30} {catalogue_coverage(train, test, fn, N_ITEMS):>23.1%}")

print()
print("Coverage is the number that makes the popularity baseline look less attractive. A")
print("recommender that only ever surfaces a handful of items has no catalogue value, does")
print("nothing for the long tail, and gives every user the same experience - even if its")
print("NDCG is the best in the table.")
print()
print("This is the first appearance of a tension that runs through the rest of the notebook:")
print("accuracy and diversity trade off, and the accuracy metric alone will always tell you")
print("to recommend the blockbuster.")

---
# Part 3 - The uncomfortable parts

Three things that are true of every offline recommender evaluation, including the one above.

## 3.1 Popularity bias, and why the baseline "wins"

§1.6 and §1.7 both ended with "most popular" on top. The obvious reading is that popularity is
simply a strong recommender. That reading is wrong, and the generator lets us prove it.

Recall that the data is **missing-not-at-random**: which cells are observed depends on item
popularity (§1.2). So popular items are over-represented *in the test set too* — and a model
that recommends popular items is rewarded for predicting **what you were shown**, not what you
would have liked.

The generator has a knob for exactly this. Turn it down and the result should reverse.

In [ ]:
print("pop_exponent controls how strongly popularity drives WHICH cells are observed.")
print("  0.0 = every user-item pair equally likely to be rated (no exposure bias)")
print("  1.0 = the realistic default used everywhere above")
print("  1.5 = an even more concentrated catalogue\n")

print(f"  {'exposure':>10} {'gini of':>9} {'popular':>10} {'factorisation':>15} {'ratio':>8}")
print(f"  {'exponent':>10} {'popularity':>9} {'NDCG@10':>10} {'NDCG@10':>15} {'':>8}")
print("  " + "-" * 60)
for e in [0.0, 0.5, 1.0, 1.5]:
    d2, _ = make_interactions(pop_exponent=e, seed=RANDOM_STATE)
    c2 = int(len(d2) * 0.8)
    tr2, te2 = d2.iloc[:c2].copy(), d2.iloc[c2:].copy()
    m2 = als(tr2, N_USERS, N_ITEMS, k=6, iters=10)
    cnt = tr2.groupby("item").size()
    pop2 = np.zeros(N_ITEMS); pop2[cnt.index.to_numpy()] = cnt.to_numpy()
    x = np.sort(pop2); nn = len(x)
    gini = float((2 * np.arange(1, nn + 1) - nn - 1) @ x / (nn * x.sum()))
    _, _, nd_pop, _ = ranking_metrics(tr2, te2, lambda u, i: pop2[i], N_ITEMS, k=10)
    _, _, nd_mf, _ = ranking_metrics(tr2, te2, lambda u, i: mf_predict(m2, u, i),
                                     N_ITEMS, k=10)
    print(f"  {e:>10.1f} {gini:>9.3f} {nd_pop:>10.4f} {nd_mf:>15.4f} "
          f"{nd_pop/max(nd_mf,1e-9):>7.1f}x")

print()
print("Read the top row first. With NO exposure bias, factorisation BEATS popularity -")
print("0.0406 against 0.0167. The personalised model was doing its job all along.")
print()
print("As exposure concentrates, popularity's apparent advantage grows to nearly 18x. Nothing")
print("about the models changed between those rows. Only the process that decided which cells")
print("you got to observe.")
print()
print("So 'the popularity baseline beats our model' is usually a statement about YOUR")
print("EVALUATION DATA, not about your model. This is the single most important thing in")
print("this notebook, and it generalises far beyond recommendation: when the data you are")
print("scored on was itself selected by a process you are trying to beat, the metric is")
print("measuring the selection.")
print()
print("What to do about it:")
print("  - Report CATALOGUE COVERAGE and long-tail metrics beside accuracy (2.2).")
print("  - Evaluate on a de-biased sample if you have one - some platforms deliberately")
print("    serve a small fraction of RANDOM recommendations to collect unbiased data.")
print("  - Use inverse-propensity weighting: weight each test interaction by 1/P(exposed).")
print("  - And treat an online A/B test as the only real answer (3.3).")

## 3.2 Which split is honest?

Random row splitting is the default in every tutorial and is wrong here for the same reason it
is wrong for time series (NB-13 §1.3): it puts a user's future on both sides of the split.

In [ ]:
rng_split = np.random.default_rng(RANDOM_STATE)
perm = rng_split.permutation(len(ratings_df))

df_sorted = ratings_df.sort_values(["user", "ts"])
last_each = df_sorted.groupby("user").tail(1)

held_users = set(rng_split.choice(N_USERS, size=int(N_USERS * 0.2), replace=False).tolist())
is_held = ratings_df["user"].isin(held_users)

protocols = {
    "random rows (the tutorial default)":
        (ratings_df.iloc[perm[:CUT]].copy(), ratings_df.iloc[perm[CUT:]].copy()),
    "chronological (what a live system faces)":
        (ratings_df.iloc[:CUT].copy(), ratings_df.iloc[CUT:].copy()),
    "leave-one-out per user (recsys standard)":
        (df_sorted.drop(last_each.index).copy(), last_each.copy()),
    "held-out USERS (pure cold start)":
        (ratings_df[~is_held].copy(), ratings_df[is_held].copy()),
}

print(f"{'protocol':<42} {'RMSE':>8} {'unseen users':>14} {'note'}")
print("-" * 92)
notes = {
    "random rows (the tutorial default)": "a user's future is in train",
    "chronological (what a live system faces)": "honest",
    "leave-one-out per user (recsys standard)": "good for ranking, ignores time",
    "held-out USERS (pure cold start)": "the hardest, and realistic for growth",
}
for name, (tr_p, te_p) in protocols.items():
    m = als(tr_p, N_USERS, N_ITEMS, k=6, iters=10)
    p = mf_predict(m, te_p["user"].to_numpy(), te_p["item"].to_numpy())
    unseen = len(set(te_p["user"]) - set(tr_p["user"]))
    print(f"{name:<42} {rmse(te_p['rating'].to_numpy(), p):>8.4f} {unseen:>14} "
          f" {notes[name]}")

print()
print("The random split reports a materially better number than the chronological one, and")
print("the 'unseen users' column is why: shuffling leaves ZERO cold-start users in the test")
print("set, while the chronological split has dozens - because users join over time.")
print()
print("The held-out-users row is the worst of all, and it is the number that matters if your")
print("product is growing: for every new signup, that is the experience.")
print()
print("Which to use:")
print("  CHRONOLOGICAL       - the default. It is what production does.")
print("  LEAVE-ONE-OUT       - the recsys literature standard, and fine for comparing")
print("                        rankers, but it ignores time entirely.")
print("  HELD-OUT USERS      - report this too if you care about new users.")
print("  RANDOM ROWS         - never.")

## 3.3 The feedback loop

The last problem has no fix inside the notebook, and you should know about it before shipping
anything.

Your recommender chooses what users see. What users see determines what they interact with.
What they interact with becomes tomorrow's training data. **The model is training on the
consequences of its own past decisions.**

The usual summary of this — "recommenders concentrate attention" — turns out to be only half
true, and the false half is the instructive one. So rather than simulate one recommender, the
cell below runs the loop under **four different serving policies** and measures where attention
ends up under each.

In [ ]:
# Simulate it: recommend, observe only what was recommended, retrain, repeat.
def gini(x):
    """0 = every item gets equal attention, 1 = one item gets all of it."""
    x = np.sort(np.asarray(x, dtype=float))
    n = len(x)
    if x.sum() == 0:
        return 0.0
    return float((2 * np.arange(1, n + 1) - n - 1) @ x / (n * x.sum()))


def simulate_feedback_loop(policy, rounds=8, top_k=20, n_active=400, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    truth_scores = TRUTH["true_score"]
    log = train.copy()
    history = []
    for r in range(rounds):
        counts = (log.groupby("item").size()
                  .reindex(range(N_ITEMS), fill_value=0).to_numpy().astype(float))
        if policy == "most popular":
            def score(uu):
                return counts
        else:
            m = als(log, N_USERS, N_ITEMS, k=6, iters=8)
            if policy == "matrix factorisation":
                def score(uu, m=m):
                    return mf_predict(m, np.full(N_ITEMS, uu), np.arange(N_ITEMS))
            else:                          # the 2.1 hybrid, with or without exploration
                hist_n = (log.groupby("user").size()
                          .reindex(range(N_USERS), fill_value=0).to_numpy())
                prior = counts / max(counts.max(), 1)
                zp = (prior - prior.mean()) / (prior.std() + 1e-9)

                def score(uu, m=m, hist_n=hist_n, zp=zp):
                    w = hist_n[uu] / (hist_n[uu] + 10.0)
                    p = mf_predict(m, np.full(N_ITEMS, uu), np.arange(N_ITEMS))
                    return w * (p - p.mean()) / (p.std() + 1e-9) + (1 - w) * zp

        users = rng.choice(N_USERS, n_active, replace=False)
        shown_all, new_rows = [], []
        for uu in users:
            shown = np.argsort(-score(uu))[:top_k]
            if policy.endswith("+ exploration"):        # 2 of the 20 slots go to random items
                shown = np.concatenate([shown[:top_k - 2],
                                        rng.choice(N_ITEMS, 2, replace=False)])
            shown_all.append(shown)
            for it in rng.choice(shown, 2, replace=False):        # they interact with 2
                rr = np.clip(np.round((truth_scores[uu, it] +
                                       rng.normal(0, 0.5)) * 2) / 2, 1, 5)
                new_rows.append({"user": uu, "item": int(it), "rating": rr, "ts": 1e6 + r})

        exposure = np.bincount(np.concatenate(shown_all), minlength=N_ITEMS)
        log = pd.concat([log, pd.DataFrame(new_rows)], ignore_index=True)
        cum = log.groupby("item").size().reindex(range(N_ITEMS), fill_value=0).to_numpy()
        history.append({"round": r, "items_shown": int((exposure > 0).sum()),
                        "gini_shown": gini(exposure), "gini_log": gini(cum)})
    return pd.DataFrame(history)


POLICIES = ["most popular", "hybrid", "hybrid + exploration", "matrix factorisation"]
loops = {p: simulate_feedback_loop(p) for p in POLICIES}

print("Each round: show 400 users their top 20, they interact with 2, retrain on the log.")
print("'items shown' = distinct items that appeared in ANYONE's top 20 in the final round.\n")
print(f"  {'serving policy':<24} {'items shown':>12} {'gini first':>11} {'gini last':>11}"
      f" {'change':>9}")
print("  " + "-" * 72)
for p in POLICIES:
    h = loops[p]
    lo, hi = h["gini_log"].iloc[0], h["gini_log"].iloc[-1]
    print(f"  {p:<24} {h['items_shown'].iloc[-1]:>12,} {lo:>11.4f} {hi:>11.4f} {hi-lo:>+9.4f}")

for p in POLICIES:
    plt.plot(loops[p]["round"], loops[p]["gini_log"], marker="o", label=p)
plt.xlabel("round"); plt.ylabel("gini of cumulative item exposure")
plt.title("Where attention ends up depends on which policy you shipped")
plt.legend(fontsize=7); plt.grid(alpha=0.3); plt.show()

In [ ]:
print("Read the table, not the folklore.\n")
print("MOST POPULAR concentrates hardest: the gini of the log climbs every round, and across")
print("400 users it shows the SAME 20 items out of 800. This is rich-get-richer in its purest")
print("form - the items it recommends gain ratings, which makes them more popular, which")
print("makes it recommend them harder.")
print()
print("The HYBRID from 2.1 concentrates too, at roughly half the rate.")
print()
print("MATRIX FACTORISATION is the surprise: attention DE-concentrates under it. That is not a")
print("virtue. It is the same fact as 1.6 - it is a weak ranker, so its top-20 lists are close")
print("to arbitrary, and scattering exposure at random flattens the distribution.")
print()
print("Now put that beside 2.2's numbers:")
print()
print(f"  {'policy':<24} {'NDCG@10':>9} {'coverage@10':>13} {'gini change':>13}")
print("  " + "-" * 62)
for p, nd, cov in [("most popular", 0.2099, "4.6%"),
                   ("hybrid", 0.2028, "17.1%"),
                   ("matrix factorisation", 0.0338, "44.2%")]:
    h = loops[p]["gini_log"]
    print(f"  {p:<24} {nd:>9.4f} {cov:>13} {h.iloc[-1]-h.iloc[0]:>+13.4f}")
print()
print("The ordering is exactly inverted. The policy the offline metric selected is the one")
print("that narrows the catalogue fastest; the policy the metric ranked last is the only one")
print("that widens it. The feedback loop is not a separate problem from 3.1 - it is the same")
print("measurement failure, running forwards in time instead of backwards.")
print()
print("EXPLORATION helps, and less than you would hope. Reserving 2 of the 20 slots for random")
print("items sharply increases how much of the catalogue is ever SHOWN, but it only SLOWS the")
print("concentration of the log - it does not reverse it. Diversity has to be paid for")
print("deliberately and continuously; a system does not drift into it.")
print()
print("What real systems do about it:")
print("  EXPLORATION      - deliberately serve some random or uncertain items. Bandit methods")
print("                     (epsilon-greedy, Thompson sampling) formalise the trade-off")
print("                     between showing the best guess and learning something.")
print("  DIVERSITY TERMS  - re-rank the top list to penalise near-duplicates, or to reserve")
print("                     slots for the tail.")
print("  UNBIASED SLICES  - keep a small random-recommendation holdout for honest evaluation")
print("                     (3.1).")
print("  ONLINE TESTING   - accept that offline metrics measure the old policy's data, and")
print("                     that an A/B test is the only measurement of the new one.")

---
# Part 4 - Tough questions

---

### Q1. What is collaborative filtering, and how does it differ from content-based recommendation?

<details><summary>Answer</summary>

**Collaborative filtering** uses only the interaction matrix: users who agreed in the past will
agree again. **Content-based** describes items by their attributes and recommends items similar
to what the user liked.

| | Collaborative | Content-based |
|---|---|---|
| Needs item metadata | **no** | yes |
| Handles a brand-new item | **no** (§1.8) | **yes** |
| Handles a brand-new user | no | partially |
| Can surprise you | **yes** — finds patterns nobody encoded | rarely — limited to the attributes |
| Fails when | data is sparse or the item is new | your attributes miss what people care about |

Collaborative filtering's real advantage is **serendipity**: it can learn that people who like
this obscure film also like that documentary, without anyone ever writing down why. A
content-based system can only ever recommend along dimensions you thought to encode.

**Every real system is a hybrid**, precisely because their failure modes are complementary:
content-based handles the new item, collaborative handles everything else. §2.1 builds a small
hybrid — a personalised score shrunk toward a popularity prior — which is the simplest useful
version of the idea.

</details>

---

### Q2. Explain matrix factorisation for recommendation.

<details><summary>Answer</summary>

Approximate the ratings matrix as a product of two thin matrices plus bias terms:

$$ \hat{r}_{ui} = \mu + b_u + b_i + p_u^\top q_i $$

Each user gets a $k$-dimensional taste vector, each item a $k$-dimensional character vector.
With $k=6$ you replace $n_u \times n_i$ unknowns with $(n_u + n_i) \times 6$ — the same
compression argument as PCA (NB-09), applied to a matrix that is mostly missing.

**Why it works:** taste is genuinely low-rank. There are not a million independent dimensions
of film preference; there are a few dozen, and everyone is a mixture.

**How it is fitted:** alternating least squares — hold $Q$ fixed and solve a ridge regression
per user, then hold $P$ fixed and solve per item. Each step is closed-form. (SGD is the other
standard route and is what Koren used for Netflix.)

**§1.5 verifies it recovers real structure**, which is only checkable because the data is
synthetic: the recovered item biases correlate with the *true* item biases at **0.958** and user
biases at **0.862**, and the predictions correlate with the true noiseless scores at **0.806**.

**Two things people get wrong:**

- **The factors are not interpretable individually.** Factorisation is unique only up to
  rotation — exactly like PCA's sign ambiguity (NB-09 §1.3). "Factor 3 is comedy" is a story.
- **More factors is not better.** §1.5 sweeps $k$; on a 4.5% dense matrix the extra capacity
  overfits. The generator used 6 factors and the sweep does not reward going far beyond that.

</details>

---

### Q3. Why is RMSE the wrong metric for a recommender?

<details><summary>Answer</summary>

**Because it measures rating prediction and a recommender does ranking.** These are different
tasks, and §1.6 measures them disagreeing about which model is better:

| model | test RMSE | NDCG@10 |
|---|---|---|
| matrix factorisation | **0.782** (best) | 0.0338 |
| random | 3.150 | 0.0166 |
| item-item CF | 3.584 | 0.1399 |
| most popular | n/a | **0.2099** (best) |

Matrix factorisation has by far the best RMSE and an NDCG barely above random. "Most popular"
has no meaningful RMSE at all — its score is a count — and the best ranking of everything tested.

**The mechanism:** RMSE is computed only over items the user *already chose to rate*. It never
asks the question a recommender faces, which is "out of 800 items, which ten?". A model can be
excellent at one and useless at the other.

**Use instead:**

- **precision@k** — of the k you recommended, how many were relevant?
- **recall@k** — of what they liked, how many did you surface?
- **NDCG@k** — as precision, but rewards putting good items nearer the top. Usually the headline.
- **MAP** — averages precision across the ranks where hits occur.

**And report non-accuracy metrics too.** §2.2 measures **catalogue coverage**: the popularity
baseline has excellent NDCG and recommends a tiny slice of the catalogue. Accuracy alone always
argues for the blockbuster.

**The historical footnote worth knowing:** the Netflix Prize was scored on RMSE, and Netflix
later said the winning ensemble was never fully deployed — partly engineering cost, and partly
that the metric was not the business problem.

</details>

---

### Q4. What is implicit feedback and why does it matter?

<details><summary>Answer</summary>

**Explicit** feedback is a rating the user deliberately gave. **Implicit** is everything else —
clicks, plays, purchases, watch time — and it is what almost every real system actually has.

Hu, Koren & Volinsky (2008) reframe the problem for it: every observed interaction is a
**positive**, every unobserved cell is a weak **negative**, and the strength of the interaction
becomes a **confidence weight** rather than a target value. The model then fits *all* cells,
which sounds impossible until you notice the "everything is zero" term factorises and can be
computed once per iteration.

**§1.7 measures the effect** — same data, same factorisation machinery, different framing:

| | test RMSE | NDCG@10 |
|---|---|---|
| explicit ALS (predict the rating) | **0.782** | 0.0338 |
| implicit ALS (predict the interaction) | 3.337 | **0.1123** |

The implicit model's RMSE is far worse — it is not predicting ratings, so the number is
meaningless — and its ranking is roughly **three times better**.

**That is the lesson:** the largest single improvement available was not a better model or more
factors. It was training on the task being measured.

**Three practical differences:**

- **There are no negatives**, only absences, and an absence is ambiguous — did they dislike it
  or never see it? That ambiguity is the whole modelling problem.
- **Confidence is not preference.** Watching a video ten times is stronger evidence than
  watching once; both are positives.
- **The scale is different.** Implicit data is far more plentiful, which is why implicit methods
  dominate in industry.

</details>

---

### Q5. Your popularity baseline beats your fancy model. What is going on?

<details><summary>Answer</summary>

**Usually your evaluation data, not your model.** §3.1 proves this by turning the cause on and
off.

The data is **missing-not-at-random**: a user rates an item only if they were shown it, and what
gets shown is driven by popularity. So popular items are over-represented in the *test set*, and
a model that recommends popular items is rewarded for predicting **what you were exposed to**
rather than what you would have liked.

§3.1 varies exactly that, holding the models constant:

| exposure ∝ popularity^e | popularity NDCG | factorisation NDCG | ratio |
|---|---|---|---|
| **0.0** (no exposure bias) | 0.0167 | **0.0406** | **0.4×** — factorisation wins |
| 0.5 | 0.0769 | 0.0337 | 2.3× |
| 1.0 (realistic) | 0.2099 | 0.0351 | 6.0× |
| 1.5 | 0.3574 | 0.0200 | 17.9× |

With no exposure bias the personalised model **wins**. Nothing about the models changed between
those rows — only the process that decided which cells you observed.

**What to do:**

1. **Report coverage and long-tail metrics** beside accuracy (§2.2). Popularity looks much worse
   on those, and rightly.
2. **Collect unbiased data.** Some platforms deliberately serve a small fraction of *random*
   recommendations purely to have an unbiased evaluation slice. It is expensive and it is the
   only clean answer.
3. **Inverse-propensity weighting** — weight each test interaction by $1/P(\text{exposed})$.
   Sound in principle, high variance in practice.
4. **A/B test.** Offline metrics evaluate the old policy's data (§3.3).

**The generalisation, which matters well beyond recommendation:** when the data you are scored
on was itself selected by the process you are trying to beat, your metric is measuring the
selection.

</details>

---

### Q6. How should you split recommender data for evaluation?

<details><summary>Answer</summary>

**Not randomly by row.** §3.2 measures four protocols:

| protocol | RMSE | unseen users in test |
|---|---|---|
| random rows | **0.654** | **0** |
| chronological | 0.782 | 33 |
| leave-one-out per user | 0.633 | 0 |
| held-out users | **1.011** | 300 — all of them |

The random split reports a materially better number, and the "unseen users" column is why: it
leaves **zero** cold-start users in the test set, while a live system meets new users
constantly. It is the same argument as NB-13 §1.3 — you have put a user's future on both sides
of the split.

**Which to use:**

- **Chronological** — the default. It is what production does, and it is the only one that
  reproduces the cold-start mix you will actually face.
- **Leave-one-out per user** — the recsys literature standard. Fine for comparing rankers, but
  it ignores time entirely, so it cannot show you drift or cold start.
- **Held-out users** — report this *as well* if your product is growing. It is the experience
  every new signup gets.
- **Random rows** — never.

**A subtlety worth naming:** even a chronological split leaks a little, because the *items* were
popular partly because of the recommender that was running at the time. §3.1 and §3.3 are about
that residue, and no split fixes it.

</details>

---

### Q7. How do you handle the cold-start problem?

<details><summary>Answer</summary>

Separate the three cases, because they need different answers.

**New user.** No history, so no $p_u$. §1.8 measures the cost: RMSE is around **60% higher** for
a brand-new user than for a user with 50+ ratings, and error falls steeply over the first
handful of ratings.

- Recommend popular items (which is why that baseline is strong).
- Onboard: ask for three favourites, or a genre.
- Use what you know anyway — country, device, referrer, time of day.
- **Blend toward the prior with a weight that grows with history**, as §2.1 does:
  $w = n/(n+k)$ gives pure popularity at $n=0$ and near-pure personalisation by $n=100$, with
  no special-casing.

**New item.** Worse for a pure collaborative model, which has literally nothing — no
interactions, no factors. This is where **content features** are not optional: derive an
initial $q_i$ from genre, text, or an embedding of the description. It is the main reason real
systems are hybrids.

**New system.** No data at all. Start content-based or editorial, and instrument everything so
that collaborative signal accumulates.

**The framing that helps:** cold start is not an edge case to patch. Because user activity
follows a power law (§1.2), *most* of your users have very little history at any given moment,
and every user starts with none. Design for it in the scoring function rather than in an
`if` statement.

</details>

---

### Q8. Item-item or user-user neighbourhood methods — and why item-item?

<details><summary>Answer</summary>

Both compute similarities and score by weighted votes. **Item-item is preferred**, for three
reasons:

1. **Item similarities are more stable.** Tastes change; what a film *is* does not. You can
   recompute the similarity matrix nightly rather than continuously.
2. **There are usually fewer items than users.** An $n_i \times n_i$ matrix is smaller, and it
   can be precomputed offline and served from a lookup.
3. **The explanation is better.** "Because you watched X" is a sentence a user accepts.
   "Because someone like you watched X" is unsettling.

**Two implementation details separate working from broken**, both in §1.4:

- **Mean-centre each user's ratings first.** Otherwise "rates everything highly" reads as
  "similar taste".
- **Shrink similarities by support.** Two items co-rated by three users can show cosine 0.99 by
  chance; multiplying by $\frac{n}{n+\lambda}$ discounts thin evidence. This is the same
  shrinkage idea as §2.1's blend and as Bayesian smoothing generally.

**Versus matrix factorisation:** neighbourhood methods are more interpretable and need no
training, but they scale as $O(n_i^2)$ in memory and do not generalise past co-occurrence.
§1.6 measures item-item well ahead of explicit factorisation on NDCG here — neighbourhood
methods being naturally ranking-oriented is part of why.

For a large catalogue, the practical answer is approximate nearest neighbours (NB-07 §3.2) or a
factorisation.

</details>

---

### Q9. What is the feedback loop and why should it worry you?

<details><summary>Answer</summary>

Your recommender decides what users see. What users see determines what they interact with.
What they interact with becomes tomorrow's training data. **The model trains on the consequences
of its own decisions.**

§3.3 simulates it — recommend a top-20, let users interact with what they were shown, retrain,
repeat — under four serving policies, and the result is sharper than the usual "recommenders
concentrate attention":

| serving policy | NDCG@10 | distinct items shown | gini of exposure, 8 rounds |
|---|---|---|---|
| most popular | **0.2099** (best) | **20** of 800 | **+0.057** |
| hybrid (§2.1) | 0.2028 | 461 | +0.032 |
| hybrid + 10% random slots | — | 667 | +0.025 |
| matrix factorisation | 0.0338 (worst) | 545 | **−0.021** |

**The ordering is exactly inverted.** The policy the offline metric selected narrows the
catalogue fastest; the policy it ranked last is the only one that widens it — and not because
factorisation is virtuous, but because §1.6 already showed it is a weak ranker, so its top-20
lists scatter exposure almost at random.

So the feedback loop is not a separate problem from Q5's popularity bias. It is the same
measurement failure running forwards in time instead of backwards.

**Why it is worse than it sounds:**

- It is **self-reinforcing**, so §3.1's popularity bias is not only an evaluation artifact but a
  production dynamic.
- It is **invisible offline**: your metrics are computed on data your own policy generated, so a
  model that narrows the catalogue can look like it is improving.
- **Exploration helps less than you would hope.** Two random slots in twenty nearly doubles how
  much of the catalogue is ever *shown* and still only **slows** the concentration of the log.
- It has **externalities** — filter bubbles, homogenisation of what gets made, and a long tail
  that never gets a chance.

**What real systems do:**

- **Exploration.** Deliberately serve some random or high-uncertainty items. Multi-armed bandits
  (ε-greedy, Thompson sampling) formalise the explore/exploit trade-off.
- **Diversity re-ranking.** Penalise near-duplicates in the top list; reserve slots for the tail.
- **An unbiased holdout** — a small fraction of random recommendations, kept purely for honest
  measurement (§3.1).
- **A/B testing**, accepting that offline evaluation measures the old policy.

</details>

---

### Q10. How does a recommender work at the scale of millions of items?

<details><summary>Answer</summary>

Not by scoring every item for every user. Production systems are **multi-stage**:

1. **Candidate generation / retrieval** — reduce millions to a few hundred, fast. Approximate
   nearest neighbours over embeddings (NB-07 §3.2 — this is exactly KNN, and it is where
   `faiss`/`hnswlib` live), plus cheap heuristics: recently popular, same category, previously
   viewed.
2. **Ranking** — score those few hundred with an expensive model (gradient boosting or a neural
   net) using rich features the retrieval stage could not afford.
3. **Re-ranking** — apply business rules, diversity (§3.3), freshness, de-duplication, and
   whatever the commercial team requires.

**Why this shape:** latency. You have tens of milliseconds. Stage 1 is sub-linear thanks to an
index; stage 2 is expensive but on a small set.

**The engineering realities the algorithm papers skip:**

- **Precompute what you can.** Item factors and the item-item matrix are computed offline;
  online you do a lookup and a dot product.
- **The item factors change slowly, the user's context changes fast.** Many systems refresh item
  embeddings nightly and update the user vector in the request.
- **Cold items need a path in.** If retrieval only proposes things with history, new items never
  get shown (§3.3), so most systems reserve capacity for exploration.

</details>

---

### Q11. Your offline NDCG improved by 15% but the A/B test showed no change. What happened?

<details><summary>Answer</summary>

This is the normal outcome, and the reasons are all in Part 3.

1. **Offline evaluation scores the old policy's data.** Your test set contains only interactions
   with items the *previous* recommender chose to show. A new model that would surface different
   items gets no credit for them — they are counted as misses because nobody was ever given the
   chance to interact (§3.1, §3.3).
2. **Popularity bias flatters whatever matches the exposure distribution** (§3.1). A model that
   shifted toward popular items may improve offline NDCG while changing nothing users notice.
3. **The offline metric is not the business metric.** NDCG@10 is not retention, revenue or
   satisfaction. A more accurate recommender that reduces diversity can be worse for retention.
4. **Position and presentation dominate.** Users click the top slot regardless. Real effects are
   often smaller than UI effects.
5. **The test may be underpowered.** Recommendation effects are typically a fraction of a
   percent; detecting that needs a lot of traffic and a long run.

**What to do:** treat offline metrics as a **filter, not a decision** — good enough to reject
bad candidates, never good enough to declare a winner. Keep an unbiased random-recommendation
slice for honest offline comparison. And define the online metric before running the test.

</details>

---

### Q12. What are the ethical problems with recommender systems that you would raise?

<details><summary>Answer</summary>

Four, and the first two are demonstrated in this notebook rather than asserted.

1. **Filter bubbles and homogenisation.** §3.3 measures the best-scoring policy narrowing the
   catalogue to **20 items out of 800**. Optimising an accuracy metric narrows what people see,
   and — because creators respond to what gets distributed — narrows what gets made.
2. **Popularity bias as an equity issue.** §3.1 shows the metric rewarding the already-popular.
   For a marketplace, that is a decision about which sellers get discovered; for media, about
   which voices are heard. It is not a neutral technical default.
3. **Optimising engagement is not optimising welfare.** Engagement is easy to measure and is a
   poor proxy for value. Content that provokes outrage engages well. If your objective is
   watch-time, you have chosen a target and should say so out loud.
4. **Privacy and inference.** Collaborative filtering infers attributes nobody disclosed —
   sexuality, pregnancy, illness, political leaning — from behaviour alone. A recommendation can
   disclose an inference to whoever is looking at the screen.

**What to do about it, concretely:**

- **Measure diversity and coverage as first-class metrics** (§2.2), not as an afterthought.
- **Build in exploration** so the tail gets a chance (§3.3).
- **Be explicit about the objective.** "We optimise watch-time" is a decision that deserves a
  decision-maker, not a default.
- **Give users control** — why am I seeing this, and how do I change it.
- **Audit by group**: are recommendations systematically worse for some users, or systematically
  suppressing some creators?

</details>

---

## Coding challenges

### Challenge 1 — build BPR and compare against implicit ALS

§1.7 uses pointwise implicit ALS. **Bayesian Personalised Ranking** (Rendle et al., 2009)
optimises the ranking directly, from triples (user, liked item, unliked item).

1. Implement BPR-MF by SGD: sample a user, a positive item they interacted with, and a random
   negative; update so the positive scores above the negative.
2. Compare NDCG@10 against implicit ALS and against `most popular` on the §1.2 data.
3. How does negative sampling affect it? Try uniform negatives against popularity-weighted
   negatives, and explain the difference using §3.1.
4. Plot NDCG against training time for both methods. Which converges faster?

---

### Challenge 2 — measure and then fix popularity bias

§3.1 demonstrates the bias. Now correct for it.

1. Implement **inverse-propensity-weighted** evaluation: estimate each item's exposure
   probability from its training frequency, then weight each test interaction by $1/\hat{p}$.
2. Recompute the §1.6 table under IPS weighting. Does the popularity baseline still win?
3. IPS is high-variance when propensities are small. Implement **clipped** IPS (cap the weights)
   and see how the ranking of models changes with the cap.
4. Now attack it in training rather than evaluation: down-weight popular items in the loss and
   measure the effect on both NDCG and catalogue coverage (§2.2). Plot the trade-off curve —
   this is the accuracy/diversity frontier, and having it as a picture is the deliverable.

---

### Challenge 3 — the full multi-stage system

Q10 describes retrieval → ranking → re-ranking. Build a small one.

1. **Retrieval:** use the implicit ALS item factors with `sklearn`'s `NearestNeighbors` to fetch
   200 candidates per user. Time it against scoring all 800 items.
2. **Ranking:** train a gradient-boosted model (NB-05) on features — ALS score, item popularity,
   item mean rating, user activity, item-item score — to predict whether the interaction
   happened. Note that this needs *negative* examples, so sample them.
3. **Re-ranking:** apply maximal marginal relevance to the top 50, trading relevance against
   dissimilarity, and measure the effect on NDCG *and* coverage.
4. Report end-to-end latency per user for the three stages, and NDCG@10 after each. Which stage
   contributed most?

---
# Part 5 - Five datasets to practise on

| # | Dataset | Scale | The skill it forces | Difficulty |
|---|---|---|---|---|
| 1 | **MovieLens 100k** | 100k ratings | The canonical starting point | ★☆☆☆☆ |
| 2 | **MovieLens 25M** | 25M ratings | Scale, and sparse-matrix discipline | ★★★☆☆ |
| 3 | **Amazon Reviews** | millions | **Implicit feedback**, extreme sparsity, text metadata | ★★★★☆ |
| 4 | **RetailRocket / e-commerce** | events | **A real funnel** — view, cart, purchase | ★★★★☆ |
| 5 | **The synthetic generator above** | you choose | Controlled experiments impossible on real data | ★★☆☆☆ |

In [ ]:
print("The generator from 1.2 is the practice dataset you can experiment with most freely.")
print("Vary its knobs and watch the difficulty change:\n")
print(f"  {'setting':<44} {'MF RMSE':>9} {'MF NDCG@10':>12}")
print("  " + "-" * 70)
for label, kwargs in [
    ("default (density 4.5%, 6 factors)", {}),
    ("sparser: density 1.5%", {"density": 0.015}),
    ("denser: density 10%", {"density": 0.10}),
    ("more complex taste: 20 factors", {"n_factors": 20}),
    ("no exposure bias (pop_exponent=0)", {"pop_exponent": 0.0}),
]:
    d, _ = make_interactions(seed=RANDOM_STATE, **kwargs)
    c = int(len(d) * 0.8)
    a, b = d.iloc[:c].copy(), d.iloc[c:].copy()
    m = als(a, N_USERS, N_ITEMS, k=6, iters=10)
    p = mf_predict(m, b["user"].to_numpy(), b["item"].to_numpy())
    _, _, nd, _ = ranking_metrics(a, b, lambda u, i: mf_predict(m, u, i), N_ITEMS, k=10)
    print(f"  {label:<44} {rmse(b['rating'].to_numpy(), p):>9.4f} {nd:>12.4f}")

print()
print("Two knobs move the numbers most, and they move them in different directions.")
print("  DENSITY controls how much evidence exists per user. Dropping to 1.5% costs")
print("     nearly half the ranking quality; doubling to 10% nearly doubles it.")
print("  TRUE COMPLEXITY (n_factors) controls how much there is to learn. At 20 true")
print("     factors a k=6 model is badly under-specified and RMSE is the worst in the")
print("     table - worse even than the sparse run.")
print("The last row is 3.1 again, arriving from the other side: removing the exposure bias")
print("IMPROVES the factorisation model's ranking while making its RMSE worse.")

### 1. MovieLens 100k — the canonical starting point

```python
# NOTE: at the time of writing the GroupLens HTTPS certificate has expired, so this
# may fail with SSLCertVerificationError. Check whether it has been fixed:
import pandas as pd
url = "https://files.grouplens.org/datasets/movielens/ml-100k/u.data"
r = pd.read_csv(url, sep="\t", names=["user", "item", "rating", "ts"])
```

943 users, 1,682 films, 100,000 ratings — every rating is 1–5 and every user has rated at least
20 films. Small enough to iterate in seconds.

1. Reproduce §1.6's table on it. Does matrix factorisation still lose on ranking while winning
   on RMSE?
2. Reproduce §1.8's cold-start curve. Real user activity is even more skewed than the
   generator's.
3. `u.item` carries genre flags — build a **content-based** recommender from them and compare
   against collaborative filtering. Then blend them (§2.1) and see whether the hybrid beats both.
4. Real timestamps span seven months. Do a chronological split (§3.2) and compare against
   leave-one-out.

---

### 2. MovieLens 25M — the same problem, at a size that forces discipline

```python
# ml-25m.zip: 25 million ratings, 162,000 users, 62,000 films
```

1. A dense 162k × 62k matrix is ~40 GB in float32. Compute that number before you start, then
   do everything with `scipy.sparse`.
2. Time item-item similarity. At 62,000 items the similarity matrix has 3.8 billion entries —
   this is where §1.4's approach stops working and Q10's retrieval stage becomes necessary.
3. Fit ALS with `k` = 8, 32, 128. Plot NDCG and fit time against `k`.
4. Use the `tags` and `genome-scores` files to build content features for cold items (Q7).

---

### 3. Amazon Reviews — implicit feedback at scale

```python
# https://cseweb.ucsd.edu/~jmcauley/datasets.html - pick one category to start
```

Purchases and reviews across product categories. The realistic setting: mostly implicit,
extremely sparse, with text.

1. Treat a review as an interaction and ignore the star rating — this is §1.7's framing. Fit
   implicit ALS.
2. Sparsity here is far below 1%. Measure how many users have fewer than 5 interactions, and
   what fraction of your evaluation they represent (§1.8).
3. Use the product text with a sentence embedding to build content vectors, and solve the
   new-item cold start (Q7).
4. Compare within-category and cross-category recommendation. Which is harder, and why?

---

### 4. RetailRocket — a real funnel

```python
# Kaggle: retailrocket-recommender-system-dataset
# events.csv has view / addtocart / transaction, with timestamps
```

The most realistic of these, because the interaction has a **type**.

1. Build separate implicit models on views, carts and purchases. Which predicts purchases best?
2. Combine them with different confidence weights (§1.7's `alpha`): a purchase is stronger
   evidence than a view. Tune the weights.
3. This data has genuine timestamps and genuine user arrival — reproduce §3.2's four protocols
   and see how much they differ on real data.
4. Measure the funnel: of your top-10 recommendations, how many were viewed, carted, purchased?
   That is the metric the business actually has.

---

### 5. The generator — for experiments real data cannot support

Use `make_interactions` to answer questions no public dataset can:

1. Reproduce §3.1's exposure-bias sweep and extend it. At what exponent does factorisation stop
   beating popularity?
2. Vary `n_factors` (the true complexity of taste) independently of `k` (the model's capacity).
   Plot the error surface. What happens when `k` is much larger than the truth?
3. Add **drift**: make user factors change slowly over time and see how much a stale model
   degrades — the recsys version of NB-13 §3.3.
4. Add a **malicious** user group that all rate one item 5 stars (a shilling attack). At what
   fraction of the user base does it succeed in promoting that item, and does item-item or
   factorisation resist it better?

---
# Part 6 - Reading the literature

An unusual field: the foundational papers came out of an open competition with a $1M prize, and
the most important recent work is about why offline evaluation misleads.

## Start here

**1. [Matrix Factorization Techniques for Recommender Systems](https://ieeexplore.ieee.org/document/5197422)** —
Yehuda Koren, Robert Bell & Chris Volinsky, *IEEE Computer* 42(8):30–37, 2009.
> **The paper that explains what won the Netflix Prize**, written for a general audience by the
> people who won it. Biases, latent factors, implicit signals, temporal dynamics — §1.5 is this
> paper. Eight pages, no heavy machinery, and still the best single introduction to the topic.

**2. [Collaborative Filtering for Implicit Feedback Datasets](http://yifanhu.net/PUB/cf.pdf)** —
Yifan Hu, Yehuda Koren & Chris Volinsky, ICDM 2008. **Free.**
> **The paper behind §1.7**, and the one that matches what real systems have. Its key move is
> separating *preference* from *confidence*, and its key trick is showing that the "all
> unobserved cells are zero" term factorises so you never build the full matrix. §1.7 implements
> it in about twenty lines and measures it tripling the ranking quality.

**3. [BPR: Bayesian Personalized Ranking from Implicit Feedback](https://arxiv.org/abs/1205.2618)** —
Steffen Rendle, Christoph Freudenthaler, Zeno Gantner & Lars Schmidt-Thieme, UAI 2009. **Free.**
> The other half of the implicit story: rather than fitting values, optimise the **ranking**
> directly from (user, positive, negative) triples. This is the conceptual bridge from §1.6's
> "you are measuring ranking" to a loss that actually optimises it, and it is Challenge 1.

## The paper behind each section

| Section | Source | Free? |
|---|---|---|
| 1.4 — item-item collaborative filtering | **Sarwar, Karypis, Konstan & Riedl**, *Item-based collaborative filtering recommendation algorithms*, WWW **2001** — [pdf](https://files.grouplens.org/papers/www10_sarwar.pdf) | ✅ |
| 1.5 — **matrix factorisation** | **Koren, Bell & Volinsky**, IEEE Computer **2009** | 🔍 |
| 1.5 — ALS for the implicit case | **Hu, Koren & Volinsky**, ICDM **2008** — [pdf](http://yifanhu.net/PUB/cf.pdf) | ✅ |
| 1.6 — **ranking metrics, not RMSE** | **Cremonesi, Koren & Turrin**, *Performance of recommender algorithms on top-N recommendation tasks*, RecSys **2010** — the paper that made this point | 🔍 |
| 1.6 — NDCG and ranking evaluation | **Järvelin & Kekäläinen**, *Cumulated gain-based evaluation of IR techniques*, ACM TOIS 20(4), **2002** | 🔍 |
| 1.7 — **BPR** | **Rendle et al.**, UAI **2009** — [arXiv](https://arxiv.org/abs/1205.2618) | ✅ |
| 1.8 — cold start | **Schein, Popescul, Ungar & Pennock**, *Methods and metrics for cold-start recommendations*, SIGIR **2002** | 🔍 |
| 3.1 — **popularity bias and MNAR data** | **Steck**, *Item popularity and recommendation accuracy*, RecSys **2011**; and **Schnabel et al.**, *Recommendations as Treatments*, ICML **2016** — [arXiv](https://arxiv.org/abs/1602.05352) | ✅ |
| 3.1 — inverse-propensity evaluation | **Schnabel et al.**, ICML **2016** | ✅ |
| 3.2 — evaluation protocol matters | **Meng, McCreadie, Macdonald & Ounis**, *Exploring Data Splitting Strategies for the Evaluation of Recommendation Models*, RecSys **2020** — [arXiv](https://arxiv.org/abs/2007.13237) | ✅ |
| 3.3 — **feedback loops** | **Chaney, Stewart & Engelhardt**, *How algorithmic confounding in recommendation systems increases homogeneity*, RecSys **2018** — [arXiv](https://arxiv.org/abs/1710.11214) | ✅ |
| Q10 — production architecture | **Covington, Adams & Sargin**, *Deep Neural Networks for YouTube Recommendations*, RecSys **2016** — [pdf](https://static.googleusercontent.com/media/research.google.com/en//pubs/archive/45530.pdf) | ✅ |
| Q11 — **offline metrics do not predict online** | **Dacrema, Cremonesi & Jannach**, *Are We Really Making Much Progress?*, RecSys **2019** — [arXiv](https://arxiv.org/abs/1907.06902) | ✅ |
| Q12 — ethics and filter bubbles | **Chaney et al.**, RecSys **2018**; and **Ekstrand et al.**, *Exploring author gender in book rating and recommendation*, RecSys **2018** | ✅ |

**Legend:** ✅ free at the link · 🔍 search the exact title on
[Google Scholar](https://scholar.google.com)

### If you read only one

**Dacrema, Cremonesi & Jannach (2019), "Are We Really Making Much Progress?"** They reproduced
18 recent neural recommendation papers and found that **most were beaten by properly tuned
simple baselines** — item-item CF and matrix factorisation. It is a short, careful, devastating
paper about how a field can appear to progress for years while comparing against weak baselines.

Read it alongside this notebook's §1.6 and §3.1, which are small demonstrations of the same
thing: the baseline is stronger than you think, and the metric is measuring something other
than what you meant.

Then **Koren, Bell & Volinsky (2009)** for the method, and **Hu et al. (2008)** for the framing
you will actually deploy.

---
# Appendix

| Symptom | Cause | Fix |
|---|---|---|
| Great RMSE, users unimpressed | RMSE measures rating prediction, not ranking (§1.6) | Report NDCG@k, precision@k, recall@k |
| The popularity baseline wins | Exposure bias in the evaluation data (§3.1) | Report coverage; IPS weighting; unbiased holdout; A/B test |
| Offline gain, no online effect | Offline data comes from the old policy (Q11, §3.3) | Treat offline as a filter, not a decision |
| Suspiciously good validation score | Random row split (§3.2) | Split chronologically; also report held-out users |
| Terrible predictions for some users | Cold start — they have no history (§1.8) | Blend toward a popularity prior with weight $n/(n+k)$ |
| A brand-new item is never recommended | Pure collaborative filtering has nothing for it (§1.8) | Content features; reserve exploration slots |
| Similarities look absurd for rare items | Computed from very few co-ratings (§1.4) | Shrink by support: $s \cdot n/(n+\lambda)$ |
| Everyone gets the same recommendations | Low catalogue coverage (§2.2) | Diversity re-ranking; measure coverage as a first-class metric |
| Recommendations narrow over time | The feedback loop (§3.3) | Exploration — bandits, random slots |
| `MemoryError` building the similarity matrix | It is $n_i \times n_i$ (§1.4) | Approximate nearest neighbours; or factorise instead |
| Latent factors have no interpretation | Factorisation is unique only up to rotation (Q2) | Do not name factors; use NB-12's methods on a downstream model |
| Implicit model has awful RMSE | It is not predicting ratings (§1.7) | Correct and expected — judge it on ranking |

## Checklist for shipping a recommender

- [ ] Am I reporting **ranking metrics** (NDCG@k) rather than RMSE (§1.6)?
- [ ] Is the **popularity baseline** in the results table (§1.3, §3.1)?
- [ ] Is **catalogue coverage** reported beside accuracy (§2.2)?
- [ ] Is the split **chronological**, and have I also measured **held-out users** (§3.2)?
- [ ] Does the scorer **degrade gracefully for cold users** (§1.8, §2.1)?
- [ ] Is there a path for **new items** to be shown at all (§1.8, §3.3)?
- [ ] Am I training on the signal I am measured on — implicit if the task is ranking (§1.7)?
- [ ] Have I thought about **exposure bias** in my evaluation data (§3.1)?
- [ ] Is there **exploration** in the serving policy (§3.3)?
- [ ] Will this be validated by an **A/B test** before anyone believes the offline number (Q11)?
- [ ] Have I stated what the system **optimises**, and is that the thing we want (Q12)?

## Where to go next

| Notebook | Why it follows |
|---|---|
| [`pca_zero_to_hero.ipynb`](pca_zero_to_hero.ipynb) | Matrix factorisation is the same low-rank idea; §1.5's rotation caveat is PCA's sign ambiguity. |
| [`knn_zero_to_hero.ipynb`](knn_zero_to_hero.ipynb) | §1.4's neighbourhood methods are KNN, and §3.2 there is the approximate search that makes retrieval possible at scale. |
| [`time_series_zero_to_hero.ipynb`](time_series_zero_to_hero.ipynb) | §3.2's "don't shuffle" argument, worked through properly. |
| `anomaly_detection_zero_to_hero.ipynb` | The last notebook in the series, and the other problem where the interesting cases are rare and the labels are missing. |

See [`README.md`](README.md) for the full roster, the reading order and the suggested paths.